# Dimension Customers
1. read the silver customers table
2. create the customer surrogate key
3. select the required columns
4. write the transformed data to gold dim_customers table

In [0]:
#Imports
from pyspark.sql import Window
from pyspark.sql.functions import row_number,col

In [0]:
customers_df = spark.read.table("neo_bank.bronze.customers")
branches_df = spark.read.table("neo_bank.gold.dim_branches")

In [0]:
window_spec=Window.orderBy("customer_id")
customers_df = customers_df.withColumn("customer_sk",row_number().over(window_spec))


In [0]:
dim_customers_df = (
    customers_df.alias("c")
        .join(
            branches_df.alias("b"),
            col("c.branch_code")==col("b.branch_code"),
            "left"
        )
)


In [0]:

dim_customers_df = (
    dim_customers_df
        .select(
            col("c.customer_sk"),
            col("c.customer_id"),
            col("c.first_name"),
            col("c.last_name"),
            col("c.date_of_birth"),
            col("c.pan_number"),
            col("c.email"),
            col("c.phone_number"),
            col("c.kyc_status"),
            col("b.branch_sk").alias("home_branch_sk")
        )
)


In [0]:
(
    dim_customers_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema","True")
        .saveAsTable("neo_bank.gold.dim_customers")
)

In [0]:
%sql
select * from neo_bank.gold.dim_customers